
# Whisper Inference Latency: Advanced Profiling Analysis

**Purpose.** This notebook produces four dissertation-grade latency analyses for the expanded MINDS-14 short banking-command benchmark:

1. **Stage-level latency decomposition** — audio loading, pad/trim, log-Mel, host-to-device transfer, encoder, and decoding.
2. **Audio duration vs inference latency** — per-utterance relationship with regression, 95% bootstrap confidence band, \(R^2\), slope, and Spearman correlation.
3. **Paired per-utterance latency improvement vs the FP32 baseline** — raincloud-style paired difference analysis with bootstrap confidence intervals and paired significance testing.
4. **Median vs tail latency** — P50/P90/P95/P99 with tail-amplification ratios.

### Statistical design

The analytical unit is **one utterance**, not one repeated timing run. Each utterance is first reduced to its median across measured runs. This avoids pseudoreplication and matches a paired experimental design when the same utterances are evaluated across configurations.

The notebook contains **no hard-coded performance results**. Every statistic and figure is computed from profiler output files at runtime.

### Visual design

The palette uses a restrained **light-blue + orange** publication style. Only the **colour idea** is taken from the supplied visual reference; no values, structure, labels, or other information from that image are used.

### Expected profiler output

Each registered profiling directory should contain:

- `raw_runs.csv`
- `per_file.csv`
- `stage_summary.csv`
- `metadata.json`

The supplied profiler already writes these files. The notebook independently validates and recomputes key quantities before analysis.


In [ ]:

from pathlib import Path
import json
import math
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy import stats
from IPython.display import display

# ------------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------------

RANDOM_SEED = 2026
BOOTSTRAP_RESAMPLES = 10_000
REGRESSION_BOOTSTRAP_RESAMPLES = 2_000
SHORT_COMMAND_MAX_SECONDS = 6.0

# Use an environment variable for automated/remote execution if needed.
PROJECT_ROOT = Path(
    os.environ.get("BANKING_VOICE_PROJECT_ROOT", Path.cwd())
).resolve()

DATASET_METADATA_PATH = PROJECT_ROOT / "data" / "minds14_banking_raw" / "metadata.csv"
RESULTS_ROOT = PROJECT_ROOT / "results"

ANALYSIS_OUTPUT_DIR = RESULTS_ROOT / "latency_advanced_analysis"
FIGURE_DIR = ANALYSIS_OUTPUT_DIR / "figures"
TABLE_DIR = ANALYSIS_OUTPUT_DIR / "tables"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Register profiling runs here.
#
# None for the baseline means:
#   use the newest results/whisper_gpu_profile_* directory.
#
# After profiling another configuration, add its exact directory:
#
# PROFILE_RUNS = {
#     "Whisper Small FP32": None,
#     "No timestamps": RESULTS_ROOT / "whisper_gpu_profile_20260809T123456Z",
# }
# ------------------------------------------------------------------

BASELINE_LABEL = "Whisper Small FP32"

PROFILE_RUNS = {
    BASELINE_LABEL: None,
}

AUTO_DISCOVERY_PATTERN = "whisper_gpu_profile_*"

# Strict set matching is recommended for the final dissertation analysis:
# every configuration should contain exactly the same <=6 s utterance set.
STRICT_SHORT_SET_MATCH = True

# ------------------------------------------------------------------
# Publication palette — colour inspiration only
# ------------------------------------------------------------------

PALETTE = {
    "sky_blue": "#78BDD7",
    "mid_blue": "#4F9FC4",
    "deep_blue": "#2E6F95",
    "navy": "#244A63",
    "orange": "#F28E2B",
    "dark_orange": "#D66B1F",
    "light_blue": "#C8E4EF",
    "pale_blue": "#EAF4F8",
    "grey": "#8A98A6",
    "light_grey": "#D9E1E7",
    "text": "#263746",
}

STAGE_COLORS = {
    "load_audio_ms": "#D8EBF3",
    "pad_trim_ms": "#BFE0EC",
    "mel_cpu_ms": "#9ACFE2",
    "h2d_transfer_ms": "#73BDD9",
    "encoder_ms": "#3E91BA",
    "decode_stage_ms": "#F28E2B",
}

STAGE_LABELS = {
    "load_audio_ms": "Audio load",
    "pad_trim_ms": "Pad / trim",
    "mel_cpu_ms": "Log-Mel",
    "h2d_transfer_ms": "H2D transfer",
    "encoder_ms": "Encoder",
    "decode_stage_ms": "Decode stage",
}

STAGE_COLUMNS = list(STAGE_LABELS.keys())

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 10.5,
    "axes.titlesize": 12,
    "axes.labelsize": 10.5,
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "legend.fontsize": 9,
    "figure.titlesize": 13,
    "axes.edgecolor": "#6C7A86",
    "axes.linewidth": 0.8,
    "grid.color": "#D8E0E6",
    "grid.linewidth": 0.65,
    "grid.alpha": 0.65,
    "savefig.facecolor": "white",
    "figure.facecolor": "white",
})

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset metadata: {DATASET_METADATA_PATH}")
print(f"Results root: {RESULTS_ROOT}")



## 1. Load and validate the short-command dataset metadata

The profiling results are joined to the prepared MINDS-14 metadata by filename. The notebook derives the **≤6 s analysis set directly from metadata**, rather than assuming a hard-coded file count.

For the current expanded benchmark, this should recover the complete selected short-command set. If a profiling run contains a different file set, the strict validation below stops the analysis rather than silently comparing unmatched utterances.


In [ ]:

def _find_first_existing_column(df, candidates, purpose):
    for column in candidates:
        if column in df.columns:
            return column
    raise KeyError(
        f"Could not find a column for {purpose}. "
        f"Tried: {candidates}. Available columns: {list(df.columns)}"
    )


def load_short_command_metadata(path, max_duration_seconds=6.0):
    if not path.exists():
        raise FileNotFoundError(
            f"Dataset metadata not found: {path}\n"
            "Run the MINDS-14 preparation/filtering pipeline first."
        )

    df = pd.read_csv(path)

    filename_col = _find_first_existing_column(
        df,
        ["file_name", "file", "filename"],
        "audio filename",
    )
    duration_col = _find_first_existing_column(
        df,
        ["duration", "duration_s", "duration_seconds"],
        "audio duration",
    )

    language_col = next(
        (c for c in ["language_variety", "language", "lang"] if c in df.columns),
        None,
    )
    intent_col = next(
        (c for c in ["intent", "intent_name", "intent_class"] if c in df.columns),
        None,
    )

    normalized = pd.DataFrame({
        "file": df[filename_col].astype(str),
        "duration_s": pd.to_numeric(df[duration_col], errors="raise"),
    })

    if language_col is not None:
        normalized["language_variety"] = df[language_col].astype(str)

    if intent_col is not None:
        normalized["intent"] = df[intent_col].astype(str)

    if normalized["file"].duplicated().any():
        duplicates = normalized.loc[
            normalized["file"].duplicated(keep=False), "file"
        ].tolist()
        raise ValueError(
            "Dataset metadata contains duplicate filenames. "
            f"Examples: {duplicates[:10]}"
        )

    if (~np.isfinite(normalized["duration_s"])).any():
        raise ValueError("Dataset metadata contains non-finite durations.")

    if (normalized["duration_s"] <= 0).any():
        raise ValueError("Dataset metadata contains non-positive durations.")

    short_df = normalized.loc[
        normalized["duration_s"] <= max_duration_seconds
    ].copy()

    if short_df.empty:
        raise ValueError(
            f"No utterances satisfy duration <= {max_duration_seconds:.2f} s."
        )

    return normalized, short_df


all_metadata, short_metadata = load_short_command_metadata(
    DATASET_METADATA_PATH,
    SHORT_COMMAND_MAX_SECONDS,
)

dataset_summary = pd.DataFrame({
    "Metric": [
        "Total prepared files",
        f"Selected files (≤{SHORT_COMMAND_MAX_SECONDS:g} s)",
        f"Not selected (>{SHORT_COMMAND_MAX_SECONDS:g} s)",
        "Selected share",
    ],
    "Value": [
        len(all_metadata),
        len(short_metadata),
        len(all_metadata) - len(short_metadata),
        f"{100 * len(short_metadata) / len(all_metadata):.2f}%",
    ],
})

display(dataset_summary)



## 2. Load, reconcile, and validate profiler outputs

This section deliberately re-checks the profiler's stored CSVs. The validation catches common silent-analysis errors such as:

- stale or mismatched `per_file.csv`,
- duplicated files,
- missing timing stages,
- non-finite or negative latency values,
- inconsistent repeated-run counts,
- a profiling run that does not contain the full ≤6 s benchmark set.

The stored `per_file.csv` medians are recomputed from `raw_runs.csv` and compared numerically before the notebook trusts them.


In [ ]:

def newest_profile_directory(results_root, pattern):
    candidates = [
        p for p in results_root.glob(pattern)
        if p.is_dir()
    ]
    if not candidates:
        return None
    return max(candidates, key=lambda p: p.stat().st_mtime)


def resolve_profile_runs(profile_runs):
    resolved = {}

    for label, configured_path in profile_runs.items():
        if configured_path is None:
            if label != BASELINE_LABEL:
                raise ValueError(
                    f"Only the baseline may use None for automatic discovery. "
                    f"Set an explicit directory for '{label}'."
                )
            path = newest_profile_directory(
                RESULTS_ROOT,
                AUTO_DISCOVERY_PATTERN,
            )
            if path is None:
                continue
        else:
            path = Path(configured_path)
            if not path.is_absolute():
                path = (PROJECT_ROOT / path).resolve()

        resolved[label] = path

    return resolved


def _assert_finite_nonnegative(df, columns, context):
    for column in columns:
        values = pd.to_numeric(df[column], errors="raise").to_numpy(dtype=float)

        if not np.isfinite(values).all():
            raise ValueError(
                f"{context}: '{column}' contains non-finite values."
            )

        if (values < 0).any():
            raise ValueError(
                f"{context}: '{column}' contains negative latency values."
            )


def load_profile_run(label, run_dir):
    required_paths = {
        "raw": run_dir / "raw_runs.csv",
        "per_file": run_dir / "per_file.csv",
        "stage_summary": run_dir / "stage_summary.csv",
        "metadata": run_dir / "metadata.json",
    }

    missing = [
        name for name, path in required_paths.items()
        if not path.exists()
    ]
    if missing:
        raise FileNotFoundError(
            f"{label}: missing expected profiler outputs in {run_dir}: {missing}"
        )

    raw = pd.read_csv(required_paths["raw"])
    stored_per_file = pd.read_csv(required_paths["per_file"])
    stored_stage_summary = pd.read_csv(required_paths["stage_summary"])

    with open(required_paths["metadata"], "r", encoding="utf-8") as f:
        metadata = json.load(f)

    required_raw_columns = {
        "file",
        "run",
        "stage_sum_ms",
        "total_profile_wall_ms",
        *STAGE_COLUMNS,
    }
    missing_raw = required_raw_columns.difference(raw.columns)

    if missing_raw:
        raise KeyError(
            f"{label}: raw_runs.csv is missing columns: {sorted(missing_raw)}"
        )

    required_per_file_columns = {
        "file",
        "stage_sum_ms",
        "total_profile_wall_ms",
        *STAGE_COLUMNS,
    }
    missing_per_file = required_per_file_columns.difference(stored_per_file.columns)

    if missing_per_file:
        raise KeyError(
            f"{label}: per_file.csv is missing columns: "
            f"{sorted(missing_per_file)}"
        )

    if raw.duplicated(["file", "run"]).any():
        raise ValueError(
            f"{label}: duplicate (file, run) rows found in raw_runs.csv."
        )

    if stored_per_file["file"].duplicated().any():
        raise ValueError(
            f"{label}: duplicate filenames found in per_file.csv."
        )

    numeric_columns = STAGE_COLUMNS + [
        "stage_sum_ms",
        "total_profile_wall_ms",
    ]

    _assert_finite_nonnegative(
        raw,
        numeric_columns,
        f"{label} raw_runs.csv",
    )
    _assert_finite_nonnegative(
        stored_per_file,
        numeric_columns,
        f"{label} per_file.csv",
    )

    # Verify stage_sum_ms against the actual row-level stage components.
    recomputed_stage_sum = raw[STAGE_COLUMNS].sum(axis=1).to_numpy(dtype=float)
    stored_stage_sum = raw["stage_sum_ms"].to_numpy(dtype=float)

    if not np.allclose(
        recomputed_stage_sum,
        stored_stage_sum,
        rtol=1e-8,
        atol=1e-6,
    ):
        max_error = np.max(np.abs(recomputed_stage_sum - stored_stage_sum))
        raise ValueError(
            f"{label}: stage_sum_ms does not equal the sum of stage columns. "
            f"Maximum absolute discrepancy: {max_error:.6f} ms."
        )

    # Recompute per-file medians independently from raw runs.
    recomputed_per_file = (
        raw.groupby("file", as_index=False)[numeric_columns]
        .median()
        .sort_values("file")
        .reset_index(drop=True)
    )

    stored_sorted = (
        stored_per_file[["file", *numeric_columns]]
        .sort_values("file")
        .reset_index(drop=True)
    )

    if list(recomputed_per_file["file"]) != list(stored_sorted["file"]):
        raise ValueError(
            f"{label}: per_file.csv filenames do not match raw_runs.csv."
        )

    if not np.allclose(
        recomputed_per_file[numeric_columns].to_numpy(dtype=float),
        stored_sorted[numeric_columns].to_numpy(dtype=float),
        rtol=1e-8,
        atol=1e-6,
    ):
        raise ValueError(
            f"{label}: per_file.csv medians do not match medians "
            "recomputed from raw_runs.csv."
        )

    # Validate repeated-run counts if metadata reports the design.
    expected_runs = metadata.get("measured_runs_per_file")
    observed_counts = raw.groupby("file").size()

    if expected_runs is not None:
        bad = observed_counts[observed_counts != int(expected_runs)]
        if not bad.empty:
            raise ValueError(
                f"{label}: repeated-run counts do not match metadata. "
                f"Expected {expected_runs}; mismatched files: {bad.index[:10].tolist()}"
            )

    recomputed_per_file["configuration"] = label
    raw = raw.copy()
    raw["configuration"] = label

    return {
        "label": label,
        "run_dir": run_dir,
        "raw": raw,
        "per_file": recomputed_per_file,
        "stored_stage_summary": stored_stage_summary,
        "metadata": metadata,
    }


resolved_runs = resolve_profile_runs(PROFILE_RUNS)

if not resolved_runs:
    print(
        "No profiler result directory is available yet.\n"
        "This is expected before profiling. The notebook is ready; "
        "run the profiler first, then rerun this notebook."
    )
    PROFILE_DATA = {}
    ANALYSIS_READY = False
else:
    PROFILE_DATA = {
        label: load_profile_run(label, path)
        for label, path in resolved_runs.items()
    }
    ANALYSIS_READY = True

    run_inventory = pd.DataFrame([
        {
            "configuration": label,
            "directory": str(data["run_dir"]),
            "profiled_files": len(data["per_file"]),
            "measured_runs_per_file": data["metadata"].get(
                "measured_runs_per_file"
            ),
            "model": data["metadata"].get("model"),
            "model_dtype": data["metadata"].get("model_dtype"),
            "gpu_name": data["metadata"].get("gpu_name"),
        }
        for label, data in PROFILE_DATA.items()
    ])

    display(run_inventory)


In [ ]:

def require_analysis_ready():
    if not ANALYSIS_READY:
        raise RuntimeError(
            "No profiling results are loaded. "
            "Run the profiler, then rerun the notebook from the top."
        )


def reconcile_with_short_metadata(profile_data, short_md, strict=True):
    require_analysis_ready()

    expected_files = set(short_md["file"])

    all_per_file = []
    all_raw = []

    for label, data in profile_data.items():
        observed_files = set(data["per_file"]["file"])

        missing = expected_files - observed_files
        extra = observed_files - expected_files

        if strict and (missing or extra):
            raise ValueError(
                f"{label}: profiled file set does not exactly match the "
                f"<= {SHORT_COMMAND_MAX_SECONDS:g} s benchmark.\n"
                f"Expected: {len(expected_files)} files\n"
                f"Observed: {len(observed_files)} files\n"
                f"Missing: {len(missing)}\n"
                f"Extra: {len(extra)}\n"
                f"Missing examples: {sorted(missing)[:8]}\n"
                f"Extra examples: {sorted(extra)[:8]}"
            )

        per_file = data["per_file"].merge(
            short_md,
            on="file",
            how="inner",
            validate="one_to_one",
        )

        raw = data["raw"].merge(
            short_md,
            on="file",
            how="inner",
            validate="many_to_one",
        )

        if per_file.empty:
            raise ValueError(
                f"{label}: no profiled files matched the short-command metadata."
            )

        all_per_file.append(per_file)
        all_raw.append(raw)

    combined_per_file = pd.concat(
        all_per_file,
        ignore_index=True,
    )
    combined_raw = pd.concat(
        all_raw,
        ignore_index=True,
    )

    return combined_per_file, combined_raw


if ANALYSIS_READY:
    per_file_all, raw_all = reconcile_with_short_metadata(
        PROFILE_DATA,
        short_metadata,
        strict=STRICT_SHORT_SET_MATCH,
    )

    coverage = (
        per_file_all.groupby("configuration")["file"]
        .nunique()
        .rename("utterances")
        .reset_index()
    )

    display(coverage)
else:
    per_file_all = pd.DataFrame()
    raw_all = pd.DataFrame()



## 3. Statistical utilities

All confidence intervals below are computed at the **utterance level** using a fixed random seed. Repeated timing runs are not treated as independent observations.

The paired comparison also applies a Holm correction to paired Wilcoxon p-values when more than one optimized configuration is compared with the baseline.


In [ ]:

def bootstrap_stat_ci(
    values,
    statistic=np.median,
    n_resamples=BOOTSTRAP_RESAMPLES,
    seed=RANDOM_SEED,
    confidence=0.95,
):
    values = np.asarray(values, dtype=float)

    if values.ndim != 1 or len(values) == 0:
        raise ValueError("bootstrap_stat_ci requires a non-empty 1-D array.")

    rng = np.random.default_rng(seed)
    alpha = (1.0 - confidence) / 2.0

    # Chunking prevents unnecessary peak memory for large experiments.
    chunk_size = min(1_000, n_resamples)
    stats_out = []

    completed = 0
    n = len(values)

    while completed < n_resamples:
        current = min(chunk_size, n_resamples - completed)
        idx = rng.integers(0, n, size=(current, n))
        sampled = values[idx]
        stats_out.append(
            np.apply_along_axis(statistic, 1, sampled)
        )
        completed += current

    boot = np.concatenate(stats_out)

    return (
        float(np.quantile(boot, alpha)),
        float(np.quantile(boot, 1.0 - alpha)),
    )


def bootstrap_regression_band(
    x,
    y,
    x_grid,
    n_resamples=REGRESSION_BOOTSTRAP_RESAMPLES,
    seed=RANDOM_SEED,
):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    x_grid = np.asarray(x_grid, dtype=float)

    if len(x) != len(y) or len(x) < 3:
        raise ValueError("Regression requires at least three paired observations.")

    rng = np.random.default_rng(seed)
    predictions = np.empty((n_resamples, len(x_grid)), dtype=float)

    for i in range(n_resamples):
        idx = rng.integers(0, len(x), len(x))
        xb = x[idx]
        yb = y[idx]

        # Extremely unlikely with real continuous durations, but guard against
        # a degenerate bootstrap sample with zero x variance.
        if np.ptp(xb) == 0:
            predictions[i, :] = np.nan
            continue

        slope, intercept = np.polyfit(xb, yb, deg=1)
        predictions[i, :] = intercept + slope * x_grid

    valid = predictions[np.isfinite(predictions).all(axis=1)]

    if len(valid) < max(100, n_resamples // 2):
        raise RuntimeError(
            "Too many degenerate regression bootstrap samples."
        )

    lower = np.quantile(valid, 0.025, axis=0)
    upper = np.quantile(valid, 0.975, axis=0)

    return lower, upper


def holm_adjust(p_values):
    p = np.asarray(p_values, dtype=float)

    if len(p) == 0:
        return p

    order = np.argsort(p)
    adjusted = np.empty_like(p)
    running_max = 0.0
    m = len(p)

    for rank, idx in enumerate(order):
        candidate = (m - rank) * p[idx]
        running_max = max(running_max, candidate)
        adjusted[idx] = min(1.0, running_max)

    return adjusted


def save_figure(fig, stem):
    png = FIGURE_DIR / f"{stem}.png"
    pdf = FIGURE_DIR / f"{stem}.pdf"

    fig.savefig(
        png,
        dpi=600,
        bbox_inches="tight",
        facecolor="white",
    )
    fig.savefig(
        pdf,
        bbox_inches="tight",
        facecolor="white",
    )

    print(f"Saved: {png}")
    print(f"Saved: {pdf}")



# Figure 1 — Stage-Level Latency Decomposition

This figure answers: **Where does the profiled latency come from?**

Each horizontal bar stacks the median utterance-level timing of the measured stages. The diamond marks the independently observed median **total profiled wall time**. This distinction is intentional: the sum of stage medians is not mathematically identical to the median of per-utterance total wall time.

For the current profiler, the `decode_stage_ms` component includes Whisper's autoregressive decoding machinery, token selection/filtering, ranking, and text construction; it should **not** be described as a pure decoder-kernel time.


In [ ]:

def build_stage_decomposition_table(per_file_df):
    rows = []

    for config, group in per_file_df.groupby("configuration", sort=False):
        row = {"configuration": config, "n": len(group)}

        for stage in STAGE_COLUMNS:
            row[stage] = float(np.median(group[stage]))

        row["stacked_stage_medians_ms"] = sum(
            row[stage] for stage in STAGE_COLUMNS
        )
        row["wall_p50_ms"] = float(
            np.median(group["total_profile_wall_ms"])
        )

        rows.append(row)

    return pd.DataFrame(rows)


def plot_stage_decomposition(per_file_df):
    require_analysis_ready()

    table = build_stage_decomposition_table(per_file_df)
    table = table.sort_values(
        "wall_p50_ms",
        ascending=True,
    ).reset_index(drop=True)

    fig_height = max(4.2, 0.75 * len(table) + 2.2)
    fig, ax = plt.subplots(figsize=(11.6, fig_height))

    y = np.arange(len(table))
    left = np.zeros(len(table), dtype=float)

    for stage in STAGE_COLUMNS:
        values = table[stage].to_numpy(dtype=float)

        ax.barh(
            y,
            values,
            left=left,
            height=0.58,
            color=STAGE_COLORS[stage],
            edgecolor="white",
            linewidth=0.6,
            label=STAGE_LABELS[stage],
            zorder=2,
        )

        # Segment labels only where there is enough visual space.
        total_width = table["stacked_stage_medians_ms"].to_numpy(dtype=float)
        share = np.divide(
            values,
            total_width,
            out=np.zeros_like(values),
            where=total_width > 0,
        )

        for i, (segment_left, value, frac) in enumerate(
            zip(left, values, share)
        ):
            if frac >= 0.075:
                ax.text(
                    segment_left + value / 2,
                    y[i],
                    f"{value:.1f}",
                    ha="center",
                    va="center",
                    fontsize=8.2,
                    color=PALETTE["text"],
                )

        left += values

    ax.scatter(
        table["wall_p50_ms"],
        y,
        marker="D",
        s=52,
        color=PALETTE["navy"],
        edgecolor="white",
        linewidth=0.7,
        zorder=5,
        label="Observed wall-time P50",
    )

    for i, row in table.iterrows():
        ax.annotate(
            f"P50 {row['wall_p50_ms']:.1f} ms",
            xy=(row["wall_p50_ms"], y[i]),
            xytext=(7, 0),
            textcoords="offset points",
            ha="left",
            va="center",
            fontsize=8.5,
            color=PALETTE["navy"],
        )

    ax.set_yticks(y)
    ax.set_yticklabels(table["configuration"])
    ax.set_xlabel("Latency (ms)")
    ax.set_ylabel("")
    ax.set_title(
        "Stage-Level Latency Decomposition",
        loc="left",
        fontweight="bold",
        pad=14,
    )

    ax.grid(axis="x", zorder=0)
    ax.grid(axis="y", visible=False)
    ax.spines[["top", "right", "left"]].set_visible(False)

    handles, labels = ax.get_legend_handles_labels()
    ax.legend(
        handles,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.13),
        ncol=4,
        frameon=False,
    )

    fig.tight_layout()
    save_figure(fig, "figure_1_stage_latency_decomposition")

    table.to_csv(
        TABLE_DIR / "figure_1_stage_latency_decomposition.csv",
        index=False,
    )

    plt.show()
    return table


if ANALYSIS_READY:
    stage_decomposition_table = plot_stage_decomposition(per_file_all)
    display(stage_decomposition_table.round(3))
else:
    print("Figure 1 is ready and will render after profiler results exist.")



# Figure 2 — Audio Duration vs Inference Latency

This analysis tests whether latency changes systematically across the **≤6 s short-command range**.

Each panel contains one point per utterance, an ordinary least-squares regression line, and a **95% bootstrap confidence band**. The annotation reports:

- slope in milliseconds of added inference latency per extra second of audio,
- 95% bootstrap confidence interval for the slope,
- \(R^2\),
- Spearman \(\rho\).

The use of utterance-level medians means repeated timing runs do not artificially inflate the sample size.


In [ ]:

def regression_statistics(x, y, seed=RANDOM_SEED):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    result = stats.linregress(x, y)
    rho, rho_p = stats.spearmanr(x, y)

    rng = np.random.default_rng(seed)
    slopes = []

    for _ in range(REGRESSION_BOOTSTRAP_RESAMPLES):
        idx = rng.integers(0, len(x), len(x))
        xb = x[idx]
        yb = y[idx]

        if np.ptp(xb) == 0:
            continue

        slopes.append(np.polyfit(xb, yb, deg=1)[0])

    slope_ci = np.quantile(slopes, [0.025, 0.975])

    return {
        "n": len(x),
        "slope_ms_per_s": float(result.slope),
        "slope_ci95_low": float(slope_ci[0]),
        "slope_ci95_high": float(slope_ci[1]),
        "intercept_ms": float(result.intercept),
        "r_squared": float(result.rvalue ** 2),
        "spearman_rho": float(rho),
        "spearman_p": float(rho_p),
    }


def plot_duration_latency_regression(per_file_df):
    require_analysis_ready()

    configs = list(
        per_file_df["configuration"].drop_duplicates()
    )
    n_configs = len(configs)

    ncols = 2 if n_configs > 1 else 1
    nrows = math.ceil(n_configs / ncols)

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(12.2, 4.5 * nrows),
        squeeze=False,
        sharex=True,
    )

    axes_flat = axes.flatten()
    stats_rows = []

    for panel_index, config in enumerate(configs):
        ax = axes_flat[panel_index]
        group = (
            per_file_df.loc[
                per_file_df["configuration"] == config
            ]
            .sort_values("duration_s")
            .copy()
        )

        x = group["duration_s"].to_numpy(dtype=float)
        y = group["total_profile_wall_ms"].to_numpy(dtype=float)

        stat_row = regression_statistics(
            x,
            y,
            seed=RANDOM_SEED + panel_index,
        )
        stat_row["configuration"] = config
        stats_rows.append(stat_row)

        x_grid = np.linspace(
            max(0.0, x.min() - 0.05),
            min(SHORT_COMMAND_MAX_SECONDS, x.max() + 0.05),
            160,
        )

        lin = stats.linregress(x, y)
        y_fit = lin.intercept + lin.slope * x_grid

        band_low, band_high = bootstrap_regression_band(
            x,
            y,
            x_grid,
            seed=RANDOM_SEED + 100 + panel_index,
        )

        ax.scatter(
            x,
            y,
            s=19,
            color=PALETTE["sky_blue"],
            alpha=0.48,
            edgecolors="none",
            rasterized=True,
            zorder=2,
        )

        ax.fill_between(
            x_grid,
            band_low,
            band_high,
            color=PALETTE["light_blue"],
            alpha=0.55,
            linewidth=0,
            zorder=1,
            label="95% bootstrap band",
        )

        ax.plot(
            x_grid,
            y_fit,
            color=PALETTE["orange"],
            linewidth=2.2,
            zorder=3,
            label="OLS fit",
        )

        ax.set_title(config, loc="left", fontweight="bold")
        ax.set_xlabel("Audio duration (s)")
        ax.set_ylabel("Median profiled wall latency (ms)")
        ax.grid(True)
        ax.spines[["top", "right"]].set_visible(False)

        annotation = (
            f"n = {stat_row['n']}\n"
            f"Slope = {stat_row['slope_ms_per_s']:.2f} ms/s\n"
            f"95% CI [{stat_row['slope_ci95_low']:.2f}, "
            f"{stat_row['slope_ci95_high']:.2f}]\n"
            f"$R^2$ = {stat_row['r_squared']:.3f}\n"
            f"Spearman $\\rho$ = {stat_row['spearman_rho']:.3f}"
        )

        ax.text(
            0.025,
            0.975,
            annotation,
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=8.7,
            bbox={
                "boxstyle": "round,pad=0.35",
                "facecolor": "white",
                "edgecolor": PALETTE["light_grey"],
                "alpha": 0.92,
            },
        )

    for unused_ax in axes_flat[n_configs:]:
        unused_ax.set_visible(False)

    fig.suptitle(
        "Audio Duration vs Inference Latency",
        x=0.08,
        ha="left",
        fontweight="bold",
    )

    legend_handles = [
        Line2D(
            [0], [0],
            marker="o",
            color="none",
            markerfacecolor=PALETTE["sky_blue"],
            markersize=6,
            label="Utterance",
        ),
        Line2D(
            [0], [0],
            color=PALETTE["orange"],
            linewidth=2.2,
            label="OLS fit",
        ),
        Patch(
            facecolor=PALETTE["light_blue"],
            edgecolor="none",
            alpha=0.55,
            label="95% bootstrap band",
        ),
    ]

    fig.legend(
        handles=legend_handles,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.005),
        ncol=3,
        frameon=False,
    )

    fig.tight_layout(rect=(0, 0.05, 1, 0.96))
    save_figure(fig, "figure_2_duration_vs_latency_regression")

    stats_table = pd.DataFrame(stats_rows)[[
        "configuration",
        "n",
        "slope_ms_per_s",
        "slope_ci95_low",
        "slope_ci95_high",
        "r_squared",
        "spearman_rho",
        "spearman_p",
    ]]

    stats_table.to_csv(
        TABLE_DIR / "figure_2_duration_latency_regression_statistics.csv",
        index=False,
    )

    plt.show()
    return stats_table


if ANALYSIS_READY:
    duration_regression_table = plot_duration_latency_regression(per_file_all)
    display(duration_regression_table.round(4))
else:
    print("Figure 2 is ready and will render after profiler results exist.")



# Figure 3 — Paired Per-Utterance Latency Improvement vs FP32 Baseline

For every utterance:

\[
\Delta \text{Latency} =
\text{Optimized latency} - \text{Baseline latency}
\]

Therefore:

- **negative values = faster than baseline**,
- **positive values = slower than baseline**.

The figure combines a violin distribution, jittered utterance-level paired differences, the median difference, and a 95% bootstrap confidence interval. Because the same utterances are compared under both configurations, the analysis is paired.

A paired Wilcoxon signed-rank test is reported as a secondary inferential check. If several configurations are compared with baseline, p-values are Holm-adjusted. Effect magnitude and consistency remain more important than p-values alone.

> This figure requires at least one optimized profiling run registered in `PROFILE_RUNS` in addition to the FP32 baseline.


In [ ]:

def paired_latency_statistics(per_file_df, baseline_label):
    configs = [
        c for c in per_file_df["configuration"].drop_duplicates()
        if c != baseline_label
    ]

    if not configs:
        return pd.DataFrame(), {}

    baseline = (
        per_file_df.loc[
            per_file_df["configuration"] == baseline_label,
            ["file", "total_profile_wall_ms"],
        ]
        .rename(columns={
            "total_profile_wall_ms": "baseline_ms"
        })
    )

    rows = []
    deltas_by_config = {}
    raw_p_values = []

    for i, config in enumerate(configs):
        optimized = (
            per_file_df.loc[
                per_file_df["configuration"] == config,
                ["file", "total_profile_wall_ms"],
            ]
            .rename(columns={
                "total_profile_wall_ms": "optimized_ms"
            })
        )

        paired = baseline.merge(
            optimized,
            on="file",
            how="inner",
            validate="one_to_one",
        )

        if len(paired) != len(baseline):
            raise ValueError(
                f"{config}: paired analysis does not contain the full "
                "baseline utterance set."
            )

        delta = (
            paired["optimized_ms"] - paired["baseline_ms"]
        ).to_numpy(dtype=float)

        deltas_by_config[config] = delta

        ci_low, ci_high = bootstrap_stat_ci(
            delta,
            statistic=np.median,
            seed=RANDOM_SEED + 300 + i,
        )

        if np.allclose(delta, 0.0):
            wilcoxon_p = 1.0
        else:
            try:
                wilcoxon_result = stats.wilcoxon(
                    delta,
                    zero_method="wilcox",
                    alternative="two-sided",
                    method="auto",
                )
                wilcoxon_p = float(wilcoxon_result.pvalue)
            except ValueError:
                wilcoxon_p = np.nan

        raw_p_values.append(wilcoxon_p)

        rows.append({
            "configuration": config,
            "n_pairs": len(delta),
            "median_delta_ms": float(np.median(delta)),
            "median_delta_ci95_low_ms": ci_low,
            "median_delta_ci95_high_ms": ci_high,
            "median_speedup_ms": float(-np.median(delta)),
            "utterances_faster_pct": float(100 * np.mean(delta < 0)),
            "utterances_slower_pct": float(100 * np.mean(delta > 0)),
            "wilcoxon_p_raw": wilcoxon_p,
        })

    stats_table = pd.DataFrame(rows)

    finite_mask = np.isfinite(stats_table["wilcoxon_p_raw"].to_numpy(dtype=float))
    adjusted = np.full(len(stats_table), np.nan)

    if finite_mask.any():
        adjusted[finite_mask] = holm_adjust(
            stats_table.loc[
                finite_mask,
                "wilcoxon_p_raw",
            ].to_numpy(dtype=float)
        )

    stats_table["wilcoxon_p_holm"] = adjusted

    return stats_table, deltas_by_config


def plot_paired_latency_improvement(per_file_df, baseline_label):
    require_analysis_ready()

    stats_table, deltas_by_config = paired_latency_statistics(
        per_file_df,
        baseline_label,
    )

    if stats_table.empty:
        print(
            "Figure 3 requires at least one optimized profiling run.\n"
            "Add the optimized result directory to PROFILE_RUNS, "
            "rerun from the configuration cell, and this figure will "
            "be generated automatically."
        )
        return stats_table

    order = (
        stats_table.sort_values(
            "median_delta_ms",
            ascending=True,
        )["configuration"]
        .tolist()
    )

    fig_height = max(4.8, 0.95 * len(order) + 2.3)
    fig, ax = plt.subplots(figsize=(11.8, fig_height))

    rng = np.random.default_rng(RANDOM_SEED + 999)

    positions = np.arange(1, len(order) + 1)

    for pos, config in zip(positions, order):
        delta = deltas_by_config[config]

        violin = ax.violinplot(
            dataset=[delta],
            positions=[pos],
            vert=False,
            widths=0.72,
            showmeans=False,
            showmedians=False,
            showextrema=False,
            points=160,
        )

        body = violin["bodies"][0]
        body.set_facecolor(PALETTE["light_blue"])
        body.set_edgecolor(PALETTE["deep_blue"])
        body.set_linewidth(0.8)
        body.set_alpha(0.70)

        jitter = rng.normal(
            loc=pos,
            scale=0.045,
            size=len(delta),
        )

        ax.scatter(
            delta,
            jitter,
            s=13,
            color=PALETTE["sky_blue"],
            alpha=0.28,
            edgecolors="none",
            rasterized=True,
            zorder=3,
        )

        row = stats_table.loc[
            stats_table["configuration"] == config
        ].iloc[0]

        ax.hlines(
            y=pos,
            xmin=row["median_delta_ci95_low_ms"],
            xmax=row["median_delta_ci95_high_ms"],
            color=PALETTE["dark_orange"],
            linewidth=3.0,
            zorder=5,
        )

        ax.scatter(
            [row["median_delta_ms"]],
            [pos],
            marker="D",
            s=60,
            color=PALETTE["orange"],
            edgecolor="white",
            linewidth=0.8,
            zorder=6,
        )

    ax.axvline(
        0,
        color=PALETTE["navy"],
        linewidth=1.2,
        linestyle="--",
        alpha=0.9,
        zorder=1,
    )

    ax.set_yticks(positions)
    ax.set_yticklabels(order)
    ax.set_xlabel(
        "Paired latency difference: optimized − FP32 baseline (ms)"
    )
    ax.set_ylabel("")
    ax.set_title(
        "Paired Per-Utterance Latency Improvement vs FP32 Baseline",
        loc="left",
        fontweight="bold",
        pad=14,
    )

    ax.grid(axis="x")
    ax.grid(axis="y", visible=False)
    ax.spines[["top", "right", "left"]].set_visible(False)

    fig.tight_layout()
    save_figure(fig, "figure_3_paired_latency_improvement")

    stats_table.to_csv(
        TABLE_DIR / "figure_3_paired_latency_statistics.csv",
        index=False,
    )

    plt.show()
    return stats_table


if ANALYSIS_READY:
    paired_latency_table = plot_paired_latency_improvement(
        per_file_all,
        BASELINE_LABEL,
    )
    if not paired_latency_table.empty:
        display(paired_latency_table.round(4))
else:
    print("Figure 3 is ready and will render after the required profiling runs exist.")



# Figure 4 — Median vs Tail Latency (P50/P90/P95/P99)

Interactive voice systems are affected by **tail latency**, not only typical latency. This figure therefore shows the complete P50→P99 span for each configuration and marks P90 and P95 explicitly.

Two additional diagnostics are exported:

- **P95/P50 tail amplification**
- **P99/P50 tail amplification**

A configuration with a competitive median but a large tail-amplification ratio may still produce noticeably inconsistent user experience.


In [ ]:

def build_tail_latency_table(per_file_df):
    rows = []

    for config, group in per_file_df.groupby("configuration", sort=False):
        values = group["total_profile_wall_ms"].to_numpy(dtype=float)

        p50, p90, p95, p99 = np.percentile(
            values,
            [50, 90, 95, 99],
        )

        rows.append({
            "configuration": config,
            "n": len(values),
            "p50_ms": float(p50),
            "p90_ms": float(p90),
            "p95_ms": float(p95),
            "p99_ms": float(p99),
            "p95_over_p50": float(p95 / p50),
            "p99_over_p50": float(p99 / p50),
        })

    return pd.DataFrame(rows)


def plot_tail_latency(per_file_df):
    require_analysis_ready()

    table = build_tail_latency_table(per_file_df)
    table = table.sort_values(
        "p50_ms",
        ascending=True,
    ).reset_index(drop=True)

    y = np.arange(len(table))

    fig_height = max(4.5, 0.78 * len(table) + 2.2)
    fig, ax = plt.subplots(figsize=(11.8, fig_height))

    for i, row in table.iterrows():
        ax.hlines(
            y=y[i],
            xmin=row["p50_ms"],
            xmax=row["p99_ms"],
            color=PALETTE["mid_blue"],
            linewidth=3.2,
            alpha=0.75,
            zorder=2,
        )

    ax.scatter(
        table["p50_ms"],
        y,
        s=62,
        marker="o",
        color=PALETTE["sky_blue"],
        edgecolor=PALETTE["deep_blue"],
        linewidth=0.8,
        zorder=4,
        label="P50",
    )

    ax.scatter(
        table["p90_ms"],
        y,
        s=52,
        marker="s",
        color=PALETTE["deep_blue"],
        edgecolor="white",
        linewidth=0.7,
        zorder=4,
        label="P90",
    )

    ax.scatter(
        table["p95_ms"],
        y,
        s=66,
        marker="D",
        color=PALETTE["orange"],
        edgecolor="white",
        linewidth=0.8,
        zorder=5,
        label="P95",
    )

    ax.scatter(
        table["p99_ms"],
        y,
        s=74,
        marker="X",
        color=PALETTE["dark_orange"],
        edgecolor="white",
        linewidth=0.8,
        zorder=5,
        label="P99",
    )

    for i, row in table.iterrows():
        ax.annotate(
            f"P95/P50 {row['p95_over_p50']:.2f}×",
            xy=(row["p99_ms"], y[i]),
            xytext=(8, 0),
            textcoords="offset points",
            ha="left",
            va="center",
            fontsize=8.3,
            color=PALETTE["grey"],
        )

    ax.set_yticks(y)
    ax.set_yticklabels(table["configuration"])
    ax.set_xlabel("Utterance-level profiled wall latency (ms)")
    ax.set_ylabel("")
    ax.set_title(
        "Median and Tail Latency",
        loc="left",
        fontweight="bold",
        pad=14,
    )

    ax.grid(axis="x")
    ax.grid(axis="y", visible=False)
    ax.spines[["top", "right", "left"]].set_visible(False)

    ax.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.12),
        ncol=4,
        frameon=False,
    )

    fig.tight_layout()
    save_figure(fig, "figure_4_tail_latency_p50_p90_p95_p99")

    table.to_csv(
        TABLE_DIR / "figure_4_tail_latency_statistics.csv",
        index=False,
    )

    plt.show()
    return table


if ANALYSIS_READY:
    tail_latency_table = plot_tail_latency(per_file_all)
    display(tail_latency_table.round(4))
else:
    print("Figure 4 is ready and will render after profiler results exist.")



## 4. Final analysis integrity report

This last section summarizes the analysis inputs and exports a compact audit table. It is intended to make the notebook easy to review during dissertation writing, viva preparation, and reproducibility checks.


In [ ]:

def build_integrity_report():
    rows = [
        {
            "check": "Short-command threshold",
            "value": f"<= {SHORT_COMMAND_MAX_SECONDS:g} s",
        },
        {
            "check": "Prepared dataset files",
            "value": len(all_metadata),
        },
        {
            "check": "Selected short-command files",
            "value": len(short_metadata),
        },
        {
            "check": "Registered profiling configurations",
            "value": len(PROFILE_DATA),
        },
        {
            "check": "Strict identical-file-set validation",
            "value": STRICT_SHORT_SET_MATCH,
        },
        {
            "check": "Bootstrap resamples",
            "value": BOOTSTRAP_RESAMPLES,
        },
        {
            "check": "Regression bootstrap resamples",
            "value": REGRESSION_BOOTSTRAP_RESAMPLES,
        },
        {
            "check": "Random seed",
            "value": RANDOM_SEED,
        },
        {
            "check": "Analytical unit",
            "value": "one utterance (median across repeated timing runs)",
        },
    ]

    report = pd.DataFrame(rows)
    report.to_csv(
        TABLE_DIR / "analysis_integrity_report.csv",
        index=False,
    )
    return report


integrity_report = build_integrity_report()
display(integrity_report)

print(f"\nFigures directory: {FIGURE_DIR}")
print(f"Tables directory: {TABLE_DIR}")

if not ANALYSIS_READY:
    print(
        "\nNotebook status: READY FOR DATA. "
        "No profiling result folder was available at execution time."
    )
else:
    print("\nNotebook status: PROFILING DATA LOADED AND VALIDATED.")



## Dissertation interpretation guardrails

When these figures are used in the dissertation:

- describe `decode_stage_ms` as the **Whisper decoding stage**, not a pure decoder-kernel measurement;
- report P50/P90/P95/P99 from **utterance-level medians**;
- use paired differences only when the **same utterance set** is available for baseline and optimized configurations;
- do not infer causality from the duration–latency regression alone;
- do not claim an optimization is beneficial from median latency alone if its tail latency or transcription accuracy deteriorates materially;
- retain the profiler `metadata.json` files alongside the exported figures/tables as reproducibility evidence.
